# 面试问题：Masked Autoencoder 为什么高遮挡率仍能学习视觉表征，如何从零实现？

可以直接复述的回答是：MAE 先把图像切成 patch，只把少量可见 patch 送进 encoder，从而节省主要计算。decoder 接收可见 latent、位置编码和 mask token，预测被遮挡 patch 的像素。损失通常只计算 masked patches，避免大量已知像素稀释训练信号。遮挡必须保留至少一个可见 patch，否则对空集合求均值会产生 NaN。真实 MAE 使用 Transformer；下面用小型线性 encoder/decoder 保留 patchify、随机 mask、只编码可见块和 masked loss 的核心机制，并真实 backward。

## 真实案例：六张 8×8 工业灰度图的遮挡重建

六张教学图分别表示水平管道、竖直管道、十字接头、方框、对角划痕和条纹。每张图切成 16 个 `2×2` patch，并固定遮挡 75%。这些是程序生成的结构化小图，不代表真实视觉数据集。

In [1]:
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量和自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警保持输出清晰
torch.set_num_threads(1)  # 固定 CPU 单线程执行
torch.manual_seed(1401)  # 固定遮挡和模型初始化
image_names = ["水平管道", "竖直管道", "十字接头", "方框结构", "对角划痕", "交替条纹"]  # 定义六种图像语义
images = torch.zeros(6, 1, 8, 8)  # 初始化六张单通道八乘八图像
images[0, 0, 3:5, :] = 1.0  # 绘制两像素高水平管道
images[1, 0, :, 3:5] = 1.0  # 绘制两像素宽竖直管道
images[2, 0, 3:5, :] = 1.0  # 绘制十字接头水平部分
images[2, 0, :, 3:5] = 1.0  # 绘制十字接头竖直部分
images[3, 0, 1:7, 1] = 1.0  # 绘制方框左边
images[3, 0, 1:7, 6] = 1.0  # 绘制方框右边
images[3, 0, 1, 1:7] = 1.0  # 绘制方框上边
images[3, 0, 6, 1:7] = 1.0  # 绘制方框下边
for index in range(8):  # 遍历对角划痕像素位置
    images[4, 0, index, index] = 1.0  # 绘制主对角线
    if index + 1 < 8:  # 检查相邻像素是否仍在图内
        images[4, 0, index, index + 1] = 0.7  # 增加一条较弱平行划痕
images[5, 0, ::2, :] = 1.0  # 绘制每隔一行的水平条纹
mask_generator = torch.Generator().manual_seed(99)  # 创建固定 patch mask 随机流
masks = torch.zeros(6, 16, dtype=torch.bool)  # 初始化六张图的十六 patch 遮挡标记
for image_index in range(6):  # 为每张图独立采样遮挡位置
    permutation = torch.randperm(16, generator=mask_generator)  # 生成可复现 patch 随机顺序
    masks[image_index, permutation[:12]] = True  # 遮挡十二个 patch 保留四个可见块
print("输入预览：name | shape | bright_pixels | masked_patches")  # 输出六图信息表头
for image_index, name in enumerate(image_names):  # 遍历六张结构化图像
    print(f"{name:6} | {tuple(images[image_index].shape)} | {int((images[image_index] > 0).sum())} | {masks[image_index].nonzero().flatten().tolist()}")  # 展示图像规模和 mask 位置
print("水平管道像素矩阵：")  # 输出首张图供直接观察
print(images[0, 0].int().tolist())  # 展示八乘八原始像素

输入预览：name | shape | bright_pixels | masked_patches
水平管道   | (1, 8, 8) | 16 | [0, 1, 2, 3, 4, 5, 8, 9, 11, 13, 14, 15]
竖直管道   | (1, 8, 8) | 16 | [1, 2, 3, 4, 5, 7, 9, 11, 12, 13, 14, 15]
十字接头   | (1, 8, 8) | 28 | [0, 1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 15]
方框结构   | (1, 8, 8) | 20 | [0, 1, 3, 4, 7, 8, 9, 10, 11, 12, 14, 15]
对角划痕   | (1, 8, 8) | 15 | [0, 1, 3, 4, 5, 6, 7, 8, 9, 11, 14, 15]
交替条纹   | (1, 8, 8) | 32 | [0, 1, 2, 3, 6, 7, 8, 9, 11, 12, 13, 14]
水平管道像素矩阵：
[[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]]


## Baseline / 基线：masked patch 用可见像素均值填充

基线保留可见 patch，所有 masked patch 填同一个可见像素均值。它不学习空间结构，但给出同一 mask 下的重建 MSE 对照。

In [2]:
def patchify(batch_images, patch_size=2):  # 手写图像到 patch 序列变换
    batch_size, channels, height, width = batch_images.shape  # 读取批量图像维度
    grid_height = height // patch_size  # 计算纵向 patch 数
    grid_width = width // patch_size  # 计算横向 patch 数
    reshaped = batch_images.reshape(batch_size, channels, grid_height, patch_size, grid_width, patch_size)  # 按 patch 网格重排像素
    permuted = reshaped.permute(0, 2, 4, 1, 3, 5)  # 把网格维移到序列前
    return permuted.reshape(batch_size, grid_height * grid_width, channels * patch_size * patch_size)  # 展平为十六个四维 patch
def unpatchify(patches, patch_size=2, height=8, width=8):  # 手写 patch 序列恢复图像
    batch_size, patch_count, patch_dimension = patches.shape  # 读取 patch 张量形状
    grid_height = height // patch_size  # 计算纵向 patch 网格
    grid_width = width // patch_size  # 计算横向 patch 网格
    reshaped = patches.reshape(batch_size, grid_height, grid_width, 1, patch_size, patch_size)  # 恢复网格和局部像素维
    permuted = reshaped.permute(0, 3, 1, 4, 2, 5)  # 把通道和空间维恢复到图像顺序
    return permuted.reshape(batch_size, 1, height, width)  # 返回批量单通道图像
patches = patchify(images)  # 将六张图切成十六个 patch
visible_mask = ~masks  # 计算每张图四个可见 patch 标记
visible_values = (patches * visible_mask.unsqueeze(-1)).sum(dim=(1, 2))  # 汇总所有可见 patch 像素
visible_counts = visible_mask.sum(dim=1) * patches.shape[-1]  # 计算可见像素数量
visible_mean = visible_values / visible_counts  # 计算每张图可见像素均值
baseline_patches = patches.clone()  # 复制原 patch 以保留可见区域
baseline_patches[masks] = visible_mean[:, None].expand(-1, 12).reshape(-1, 1).expand(-1, 4)  # 用每图均值填充十二个 masked patch
baseline_masked_mse = float(((baseline_patches[masks] - patches[masks]) ** 2).mean())  # 只在未知 patch 上计算基线误差
print("patch shape：", tuple(patches.shape), "每图 visible：", visible_mask.sum(dim=1).tolist())  # 展示 patchify 和遮挡规模
print("各图可见像素均值：", [round(value, 3) for value in visible_mean.tolist()])  # 展示基线填充值
print(f"均值填充 masked MSE={baseline_masked_mse:.6f}")  # 输出同数据基线指标

patch shape： (6, 16, 4) 每图 visible： [4, 4, 4, 4, 4, 4]
各图可见像素均值： [0.375, 0.25, 0.375, 0.25, 0.169, 0.5]
均值填充 masked MSE=0.209599


## 核心实现：只聚合 visible token 的小型 MAE

encoder 将四维 patch 投影到十二维 latent，并加位置向量；mask 后只对可见 token 求均值。decoder 使用全局 latent 与每个位置 embedding 重建所有 patch，训练损失只读取 masked 索引。

In [3]:
class TinyMAE(torch.nn.Module):  # 定义保留 MAE 数据流的轻量编码解码器
    def __init__(self, patch_dimension=4, latent_dimension=12, patch_count=16):  # 初始化 patch 投影和位置参数
        super().__init__()  # 初始化 PyTorch 模块基类
        self.encoder_weight = torch.nn.Parameter(torch.randn(patch_dimension, latent_dimension) * 0.15)  # 创建 patch encoder 权重
        self.encoder_bias = torch.nn.Parameter(torch.zeros(latent_dimension))  # 创建 encoder 偏置
        self.position = torch.nn.Parameter(torch.randn(patch_count, latent_dimension) * 0.05)  # 创建可学习 patch 位置向量
        self.decoder_weight = torch.nn.Parameter(torch.randn(latent_dimension, patch_dimension) * 0.15)  # 创建 latent 到像素 decoder 权重
        self.decoder_bias = torch.nn.Parameter(torch.zeros(patch_dimension))  # 创建 decoder 像素偏置
    def forward(self, patch_batch, patch_mask):  # 定义只编码可见 patch 的前向传播
        embedded = torch.tanh(patch_batch @ self.encoder_weight + self.encoder_bias + self.position.unsqueeze(0))  # 编码像素并注入位置
        visible = (~patch_mask).unsqueeze(-1).float()  # 把可见标记扩展到 latent 维
        visible_count = visible.sum(dim=1).clamp(min=1.0)  # 为全遮挡异常提供安全分母
        global_latent = (embedded * visible).sum(dim=1) / visible_count  # 只聚合可见 token 表征
        decoder_input = torch.tanh(global_latent.unsqueeze(1) + self.position.unsqueeze(0))  # 将全局语义与每个 patch 位置结合
        prediction = decoder_input @ self.decoder_weight + self.decoder_bias  # 预测十六个 patch 的四个像素
        return prediction, global_latent, embedded  # 返回重建、可见摘要和全部编码中间量
mae_model = TinyMAE()  # 创建待训练小型 MAE
training_trace = []  # 保存关键步 masked loss 和梯度范数
for step in range(1, 501):  # 对固定六图遮挡任务训练五百步
    predicted_patches, global_latent, embedded = mae_model(patches, masks)  # 真实执行 MAE forward
    loss = ((predicted_patches[masks] - patches[masks]) ** 2).mean()  # 只计算 masked patch 像素损失
    loss.backward()  # 真实执行 backward 得到 encoder、位置和 decoder 梯度
    gradient_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in mae_model.parameters()))  # 计算全局梯度范数
    with torch.no_grad():  # 关闭手写参数更新的计算图
        for parameter in mae_model.parameters():  # 遍历 MAE 全部可训练参数
            parameter -= 0.05 * parameter.grad  # 使用固定学习率执行 SGD 更新
            parameter.grad.zero_()  # 清空本步梯度
    if step in {1, 10, 100, 500}:  # 保存最有解释力的收敛节点
        training_trace.append((step, float(loss), float(gradient_norm)))  # 记录 masked loss 和真实梯度
with torch.no_grad():  # 进入重建评估阶段
    final_patches, final_latent, final_embedded = mae_model(patches, masks)  # 计算训练后所有 patch 预测
mae_masked_mse = float(((final_patches[masks] - patches[masks]) ** 2).mean())  # 计算主方案 masked MSE
print("step | masked_loss | grad_norm")  # 输出训练轨迹表头
for item in training_trace:  # 遍历四个训练节点
    print(f"{item[0]:4d} | {item[1]:.6f} | {item[2]:.6f}")  # 展示真实 forward/backward 收敛
print("中间张量：embedded", tuple(final_embedded.shape), "visible latent", tuple(final_latent.shape), "prediction", tuple(final_patches.shape))  # 展示核心张量形状
print(f"MAE masked MSE={mae_masked_mse:.6f}")  # 输出主方案重建指标

step | masked_loss | grad_norm
   1 | 0.330560 | 0.425631
  10 | 0.269856 | 0.306687
 100 | 0.190840 | 0.063080
 500 | 0.142854 | 0.035827
中间张量：embedded (6, 16, 12) visible latent (6, 12) prediction (6, 16, 4)
MAE masked MSE=0.142790


## 六图结果表与重建像素

In [4]:
combined_patches = patches.clone()  # 从原 patch 创建可视化重建副本
combined_patches[masks] = final_patches[masks]  # 只用模型输出替换 masked 区域
reconstructed_images = unpatchify(combined_patches)  # 把 patch 序列还原为六张图像
print("image | baseline_masked_MSE | MAE_masked_MSE")  # 输出逐图重建结果表头
per_image_mae = []  # 收集每张图 masked MSE
for image_index, name in enumerate(image_names):  # 遍历六张结构化图像
    baseline_error = float(((baseline_patches[image_index][masks[image_index]] - patches[image_index][masks[image_index]]) ** 2).mean())  # 计算当前图均值填充误差
    mae_error = float(((final_patches[image_index][masks[image_index]] - patches[image_index][masks[image_index]]) ** 2).mean())  # 计算当前图模型误差
    per_image_mae.append(mae_error)  # 保存逐图主方案误差
    print(f"{name:6} | {baseline_error:.6f} | {mae_error:.6f}")  # 展示同 mask 下的逐图对照
print("水平管道重建矩阵（四舍五入）：")  # 输出真实重建图标题
print(torch.round(reconstructed_images[0, 0] * 10.0).div(10.0).tolist())  # 展示首图可见与预测像素组合

image | baseline_masked_MSE | MAE_masked_MSE
水平管道   | 0.192708 | 0.143026
竖直管道   | 0.187500 | 0.150646
十字接头   | 0.255208 | 0.218840
方框结构   | 0.229167 | 0.194520
对角划痕   | 0.143008 | 0.138056
交替条纹   | 0.250000 | 0.011652
水平管道重建矩阵（四舍五入）：
[[0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.30000001192092896, 0.20000000298023224, 0.10000000149011612, 0.10000000149011612], [0.20000000298023224, 0.4000000059604645, 0.30000001192092896, 0.4000000059604645, 0.4000000059604645, 0.20000000298023224, 0.30000001192092896, 0.20000000298023224], [0.20000000298023224, 0.20000000298023224, 0.30000001192092896, 0.4000000059604645, 0.0, 0.0, 0.0, 0.0], [0.30000001192092896, 0.4000000059604645, 0.30000001192092896, 0.5, 1.0, 1.0, 1.0, 1.0], [0.30000001192092896, 0.5, 0.30000001192092896, 0.4000000059604645, 1.0, 1.0, 0.30000001192092896, 0.30000001192092896], [0.20000000298023224, 0.30000001192092896, 0.20000000298023224, 0.30000001192092896, 0.0, 0.0, 0.400000005960464

## 失败案例与修正：100% mask 对空 token 求均值

朴素代码直接 `visible_tokens.mean()`，当没有可见 patch 时得到 NaN。主模型用可见计数 `clamp(min=1)`；生产 MAE 通常从采样器层面保证至少保留一个 token。

In [5]:
all_masked = torch.ones(1, 16, dtype=torch.bool)  # 构造百分之百遮挡边界输入
empty_visible_tokens = final_embedded[:1][~all_masked]  # 按朴素索引得到空 token 张量
naive_empty_mean = empty_visible_tokens.mean()  # 复现对空集合求均值产生 NaN
with torch.no_grad():  # 关闭安全前向梯度记录
    safe_prediction, safe_latent, safe_embedded = mae_model(patches[:1], all_masked)  # 使用 clamp 分母执行全遮挡前向
print("空 visible token shape：", tuple(empty_visible_tokens.shape))  # 展示失败输入确实为空
print("朴素 mean：", float(naive_empty_mean), "是否有限：", bool(torch.isfinite(naive_empty_mean)))  # 展示 NaN 失败
print("安全 latent 是否有限：", bool(torch.isfinite(safe_latent).all()), "预测范围：", (float(safe_prediction.min()), float(safe_prediction.max())))  # 展示数值门禁修正

空 visible token shape： (0, 12)
朴素 mean： nan 是否有限： False
安全 latent 是否有限： True 预测范围： (0.03290031850337982, 0.39531245827674866)


## 结果解读

均值填充无法利用 patch 位置和可见结构；TinyMAE 的位置向量与可见 latent 共同降低 masked MSE。训练和结果始终只在未知 patch 上评分。全遮挡反例说明 mask sampler 也是模型正确性的一部分，不能假设可见 token 永远存在。

## 生产边界

教学 encoder 是可见 token 均值而非 Transformer，没有多尺度纹理、归一化像素或随机增强。真实 MAE 需只把 visible token 送入 encoder 以获得计算收益，并用独立验证图和线性探针评估表征，不能以六图重建误差宣称下游泛化。还需处理分布式 mask seed、混合精度和 checkpoint。

## 最小回归测试

In [6]:
assert len(images) >= 6 and patches.shape == (6, 16, 4)  # 保证六图真实 patchify 结果正确
assert all(int(count) == 4 for count in visible_mask.sum(dim=1))  # 保证每张图实际保留四个可见 patch
assert training_trace[-1][1] < training_trace[0][1]  # 保证真实 backward 更新降低 masked loss
assert mae_masked_mse < baseline_masked_mse  # 保证同数据主方案优于均值填充基线
assert all(torch.isfinite(torch.tensor(per_image_mae)))  # 保证六图重建指标均为有限值
assert not bool(torch.isfinite(naive_empty_mean))  # 保证全遮挡空均值失败真实复现
assert torch.isfinite(safe_prediction).all() and torch.isfinite(safe_latent).all()  # 保证安全分母修复全遮挡数值异常